# Experiment 5: Cross-Encoder Comparison & Analysis

## Experimental Design Summary

10 runs total: 5 encoders $\times$ 2 regularization conditions.

| Encoder | Group $G$ | Invariance | Augmentation |
|---------|-----------|------------|-------------|
| ViT-L/16 (baseline) | None | None | Block P |
| ViT-L/16 (augmented) | None | Approximate | Block P + G |
| Octic ViT | $D_4$ | Exact | Block P |
| Shift-Eq ViT | $\mathbb{Z}^2$ | Exact | Block P |
| Harmformer | $SE(2)$ | Exact | Block P |

## Group Hierarchy

$$D_4 \subset SE(2), \quad \mathbb{Z}^2 \subset SE(2)$$

- $D_4$: 8 elements (rotations $\{0^\circ, 90^\circ, 180^\circ, 270^\circ\}$ + reflections)
- $\mathbb{Z}^2$: infinite (cyclic pixel shifts)
- $SE(2) = SO(2) \ltimes \mathbb{R}^2$: continuous roto-translations

## Core Theoretical Properties

**Prop 1 (Fusion symmetry):** $h_\psi(z_1, z_2) = h_\psi(z_2, z_1)$

**Prop 2 (Isometry invariance):** $\mathcal{L}_{\mathrm{contr}}^{L^2}(Uz_1, Uz_2, y) = \mathcal{L}_{\mathrm{contr}}^{L^2}(z_1, z_2, y)$ for orthogonal $U$

**Prop 3 (Invariance propagation):** If $\varphi_\theta$ is $G$-invariant, then $f_\theta(g \cdot x_1, g \cdot x_2) = f_\theta(x_1, x_2)$

## Metrics

- **Primary:** FPR at F1-optimal threshold, Recall per transformation class
- **Secondary:** F1 score, t-SNE separability

In [1]:
import sys
sys.path.insert(0, '..')

import json
import os
import torch
import yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 12

from src.utils.seed import set_seed
from src.data_module.domainnet_dataset import DomainNetEvalDataset
from src.encoders import EncoderFactory
from src.models.siamese import SiameseNet
from src.validation.evaluator import DomainNetEvaluator
from src.validation.tsne import compute_tsne_embeddings, plot_tsne
from torch.utils.data import DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'

octic_vits not installed. OcticViTEncoder will fall back to ViT-L/16. Install via: pip install git+https://github.com/nordstrom/octic-vit.git
shift_eq_vit not installed. ShiftEquivariantEncoder will use shift-averaging approximation over ViT-L/16.


In [2]:
from src.utils import verify_all_encoders, report_equivariance

results = verify_all_encoders(device=device, tol=1e-3)
print(report_equivariance(results))

shift_eq_vit not installed — Z2 test uses diagonal-averaging fallback.
octic_vits not installed — D4 verification skipped.


Encoder                   Group                        Mean err      Max err   Status
------------------------------------------------------------------------------------
harmformer                SO(2)                      1.3590e+00   2.3782e+00   FAILED
shift_eq_vit              Z2                         1.5937e-06   1.6456e-06   PASSED
vit_baseline              D4 (negative control)      1.6130e+00   1.7669e+00      n/a


## Load All Reports

In [ ]:
RUN_NAMES = [
    'vit_baseline_no_reg', 'vit_baseline_reg',
    'vit_aug_no_reg', 'vit_aug_reg',
    'octic_vit_no_reg', 'octic_vit_reg',
    'shift_eq_no_reg', 'shift_eq_reg',
    'harmformer_no_reg', 'harmformer_reg',
]

reports = {}
for name in RUN_NAMES:
    path = f'../outputs/logs/{name}_report.json'
    if os.path.exists(path):
        with open(path) as f:
            reports[name] = json.load(f)
        print(f'Loaded: {name}')
    else:
        print(f'Missing: {name}')

print(f'\nLoaded {len(reports)}/{len(RUN_NAMES)} reports')

## Table 3: FPR by Encoder and Regularization

Primary metric comparing all 10 runs.

In [ ]:
# Build FPR comparison table
encoder_groups = [
    ('ViT Baseline', 'vit_baseline'),
    ('ViT Augmented', 'vit_aug'),
    ('Octic ViT ($D_4$)', 'octic_vit'),
    ('Shift-Eq ViT ($\\mathbb{Z}^2$)', 'shift_eq'),
    ('Harmformer ($SE(2)$)', 'harmformer'),
]

def _fmt_metric(v):
    if v == "N/A" or v is None:
        return str(v)
    return repr(float(v))


print(f'{"Encoder":<30} {"FPR (no reg)":>24} {"FPR (reg)":>24} {"F1 (no reg)":>24} {"F1 (reg)":>24}')
print('-' * 130)
for label, prefix in encoder_groups:
    noreg = reports.get(f'{prefix}_no_reg', {})
    reg = reports.get(f'{prefix}_reg', {})
    print(
        f'{label:<30} {_fmt_metric(noreg.get("fpr", "N/A")):>24} {_fmt_metric(reg.get("fpr", "N/A")):>24} '
        f'{_fmt_metric(noreg.get("f1", "N/A")):>24} {_fmt_metric(reg.get("f1", "N/A")):>24}'
    )

## Figure 4: FPR Grouped Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

labels = ['ViT Base', 'ViT Aug', 'Octic ($D_4$)', 'Shift-Eq ($\\mathbb{Z}^2$)', 'Harmformer ($SE(2)$)']
prefixes = ['vit_baseline', 'vit_aug', 'octic_vit', 'shift_eq', 'harmformer']

fpr_noreg = [reports.get(f'{p}_no_reg', {}).get('fpr', 0) for p in prefixes]
fpr_reg = [reports.get(f'{p}_reg', {}).get('fpr', 0) for p in prefixes]

x = np.arange(len(labels))
width = 0.35

bars1 = ax.bar(x - width/2, fpr_noreg, width, label='$\\lambda=0$', color='#4daf4a', alpha=0.8)
bars2 = ax.bar(x + width/2, fpr_reg, width, label='$\\lambda=0.1$', color='#377eb8', alpha=0.8)

ax.set_ylabel('FPR', fontsize=14)
ax.set_title('False Positive Rate by Encoder', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.legend(fontsize=12)
ax.set_ylim(0, max(max(fpr_noreg), max(fpr_reg)) * 1.3 + 0.01)

fig.tight_layout()
fig.savefig('../outputs/figures/fpr_comparison.pdf', dpi=300)
plt.show()

## Figure 5: Recall Stratified by Transformation Class

In [ ]:
transform_classes = ['CJ', 'GN', 'GS', 'R90', 'R180', 'R270', 'WM', 'COMBO']
encoder_labels = ['ViT Base', 'ViT Aug', 'Octic', 'Shift-Eq', 'Harmformer']
run_keys = ['vit_baseline_reg', 'vit_aug_reg', 'octic_vit_reg', 'shift_eq_reg', 'harmformer_reg']
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(transform_classes))
width = 0.15

for i, (key, label, color) in enumerate(zip(run_keys, encoder_labels, colors)):
    report = reports.get(key, {})
    per_tfm = report.get('per_transform', {})
    recalls = [per_tfm.get(tc, {}).get('recall', 0) for tc in transform_classes]
    ax.bar(x + i * width, recalls, width, label=label, color=color, alpha=0.85)

ax.set_ylabel('Recall', fontsize=14)
ax.set_title('Recall by Transformation Class (with $L^2$ regularizer)', fontsize=14)
ax.set_xticks(x + width * 2)
ax.set_xticklabels(transform_classes, fontsize=12)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)

fig.tight_layout()
fig.savefig('../outputs/figures/recall_by_transform.pdf', dpi=300)
plt.show()

## Figure 6: t-SNE Comparison (All 5 Encoders)

Side-by-side t-SNE plots showing domain separation quality across encoders.

In [ ]:
with open('../configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

domainnet_ds = DomainNetEvalDataset(
    root=cfg['data']['domainnet_root'],
    domains=cfg['data']['domainnet_domains'],
    pairs_per_domain=cfg['data']['pairs_per_domain'],
)
tsne_loader = DataLoader(domainnet_ds, batch_size=32, shuffle=False, num_workers=2)

encoder_configs = [
    ('ViT Baseline', 'vit_baseline', {'freeze': True}),
    ('ViT Augmented', 'vit_augmented', {'freeze': True}),
    ('Octic ViT ($D_4$)', 'octic_vit', {'freeze': True}),
    ('Shift-Eq ViT ($\\mathbb{Z}^2$)', 'shift_eq_vit', {'freeze': False}),
    ('Harmformer ($SE(2)$)', 'harmformer', {'freeze': True, 'n_rots': 8}),
]

fig, axes = plt.subplots(1, 5, figsize=(25, 5))

from src.validation.tsne import DOMAIN_COLORS

for idx, (title, enc_name, kwargs) in enumerate(encoder_configs):
    # Load best checkpoint if available
    ckpt_prefix = enc_name.replace('vit_augmented', 'vit_aug').replace('vit_baseline', 'vit_baseline')
    ckpt_path = f'../outputs/checkpoints/{ckpt_prefix}_reg/{ckpt_prefix}_reg_best.pt'
    
    encoder = EncoderFactory(enc_name, **kwargs)
    model = SiameseNet(encoder, hidden_dim=512, dropout=0.1)
    
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
    
    model = model.to(device)
    coords, domains = compute_tsne_embeddings(model, tsne_loader, device=device)
    
    ax = axes[idx]
    for domain in sorted(set(domains)):
        mask = np.array([d == domain for d in domains])
        color = DOMAIN_COLORS.get(domain, '#333')
        ax.scatter(coords[mask, 0], coords[mask, 1], c=color, label=domain, alpha=0.6, s=10)
    ax.set_title(title, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
    if idx == 0:
        ax.legend(fontsize=8, markerscale=2)

fig.suptitle('t-SNE Encoder Embeddings on DomainNet', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig('../outputs/figures/tsne_all_encoders.pdf', dpi=300, bbox_inches='tight')
plt.show()

## Table 4: Per-Transform Recall (All Encoders)

In [ ]:
transform_classes = ['CJ', 'GN', 'GS', 'R90', 'R180', 'R270', 'WM', 'COMBO']

header = f'{"Run":<25}' + ''.join(f'{tc:>24}' for tc in transform_classes)
print(header)
print('-' * len(header))

for name in RUN_NAMES:
    report = reports.get(name, {})
    per_tfm = report.get('per_transform', {})
    row = f'{name:<25}'
    for tc in transform_classes:
        recall = per_tfm.get(tc, {}).get('recall', float('nan'))
        row += f'{repr(float(recall)) if recall == recall else repr(recall):>24}'
    print(row)